In [ ]:
import os
os.chdir("..")
%pwd

In [8]:
%pwd

'e:\\MLOPS\\bid_bot_detection'

In [9]:
from dotenv import load_dotenv
load_dotenv()

import mlflow
mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
print("Tracking URI:", mlflow.get_tracking_uri())

e:\MLOPS\bid_bot_detection\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tracking URI: https://dagshub.com/fakhrulfaiz/bid-bot-detection.mlflow


In [10]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    train_data_path: Path
    test_data_path: Path
    model_name: str
    target_column: str


@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    metric_file_name: Path
    target_column: str

In [11]:
from src.datascience.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH, SCHEMA_FILE_PATH
from src.datascience.utils.common import read_yaml, create_directories, save_json

In [12]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
        schema_filepath=SCHEMA_FILE_PATH,
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        create_directories([config.root_dir])
        return ModelTrainerConfig(
            root_dir=Path(config.root_dir),
            train_data_path=Path(config.train_data_path),
            test_data_path=Path(config.test_data_path),
            model_name=config.model_name,
            target_column=self.schema.TARGET_COLUMN.name,
        )

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        create_directories([config.root_dir])
        return ModelEvaluationConfig(
            root_dir=Path(config.root_dir),
            test_data_path=Path(config.test_data_path),
            model_path=Path(config.model_path),
            metric_file_name=Path(config.metric_file_name),
            target_column=self.schema.TARGET_COLUMN.name,
        )

In [13]:
import joblib
import mlflow.sklearn
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)
from src.datascience import logger

In [15]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train(self):
        train_df = pd.read_csv(self.config.train_data_path)
        test_df  = pd.read_csv(self.config.test_data_path)

        X_train = train_df.drop(columns=[self.config.target_column])
        y_train = train_df[self.config.target_column]
        X_test  = test_df.drop(columns=[self.config.target_column])
        y_test  = test_df[self.config.target_column]

        neg = (y_train == 0).sum()
        pos = (y_train == 1).sum()
        scale_pos_weight = neg / pos
        logger.info(f"neg: {neg} | pos: {pos} | scale_pos_weight: {scale_pos_weight:.2f}")

        params = read_yaml(PARAMS_FILE_PATH)
        param_grid = {
            'n_estimators':     list(params.GridSearchCV.n_estimators),
            'max_depth':        list(params.GridSearchCV.max_depth),
            'learning_rate':    list(params.GridSearchCV.learning_rate),
            'subsample':        list(params.GridSearchCV.subsample),
            'colsample_bytree': list(params.GridSearchCV.colsample_bytree),
        }

        xgb = XGBClassifier(
            scale_pos_weight=scale_pos_weight,
            eval_metric='logloss',
            random_state=42,
            verbosity=0,
        )

        gs = GridSearchCV(
            estimator=xgb,
            param_grid=param_grid,
            cv=params.GridSearchCV.cv,
            scoring=params.GridSearchCV.scoring,
            n_jobs=params.GridSearchCV.n_jobs,
            verbose=1,
        )

        logger.info("Starting GridSearchCV")
        gs.fit(X_train, y_train)

        logger.info(f"Best params:     {gs.best_params_}")
        logger.info(f"Best CV ROC AUC: {gs.best_score_:.4f}")

        model_path = Path(self.config.root_dir) / self.config.model_name
        joblib.dump(gs.best_estimator_, model_path)
        logger.info(f"Model saved -> {model_path}")

        return gs.best_estimator_

In [29]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def evaluate(self, thresholds=None):

        if thresholds is None:
            thresholds = [0.3, 0.4, 0.5, 0.6]

        test_df = pd.read_csv(self.config.test_data_path)

        X_test = test_df.drop(columns=[self.config.target_column])
        y_test = test_df[self.config.target_column]

        model = joblib.load(self.config.model_path)
        y_test  = y_test.astype(int)
        y_prob = model.predict_proba(X_test)[:, 1]
        roc_auc = roc_auc_score(y_test, y_prob)

        results = {}

        with mlflow.start_run():

            mlflow.log_params(model.get_params())
            mlflow.log_metric("roc_auc", round(roc_auc, 4))

            for t in thresholds:
                y_pred = (y_prob >= t).astype(int)

                report = classification_report(y_test, y_pred, output_dict=True)
                cm = confusion_matrix(y_test, y_pred)

                # CORRECT — fall back to 0.0 if class '1' is absent
                precision = report.get('1', {}).get('precision', 0.0)
                recall    = report.get('1', {}).get('recall',    0.0)
                f1        = report.get('1', {}).get('f1-score',  0.0)
                accuracy = accuracy_score(y_test, y_pred)

                # Log metrics
                # Add this line before the log_metric block
                t_key = str(t).replace('.', '_')

                # Then change the 4 log_metric lines to use t_key instead of t
                mlflow.log_metric(f'accuracy_{t_key}',  round(accuracy,  4))
                mlflow.log_metric(f'precision_{t_key}', round(precision, 4))
                mlflow.log_metric(f'recall_{t_key}',    round(recall,    4))
                mlflow.log_metric(f'f1_{t_key}',        round(f1,        4))

                results[f"threshold_{t}"] = {
                    "accuracy": round(accuracy, 4),
                    "precision": round(precision, 4),
                    "recall": round(recall, 4),
                    "f1": round(f1, 4),
                    "confusion_matrix": cm.tolist()
                }

            mlflow.sklearn.log_model(model, artifact_path="model")

        return {
            "roc_auc": round(roc_auc, 4),
            "threshold_results": results
        }

    def _plot_roc(self, y_test, y_prob, roc_auc: float) -> Path:
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        plt.figure(figsize=(8, 6))
        plt.plot(fpr, tpr, color='darkorange', lw=2,
                 label=f'ROC curve (AUC = {roc_auc:.4f})')
        plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--',
                 label='Random classifier')
        plt.fill_between(fpr, tpr, alpha=0.1, color='darkorange')
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate', fontsize=13)
        plt.ylabel('True Positive Rate', fontsize=13)
        plt.title('ROC Curve — Bid Bot Detection', fontsize=14)
        plt.legend(loc='lower right', fontsize=12)
        plt.tight_layout()

        roc_path = Path(self.config.root_dir) / 'roc_curve.png'
        plt.savefig(roc_path, dpi=150)
        plt.show()
        logger.info(f"ROC curve saved -> {roc_path}")
        return roc_path

In [17]:
# --- Train ---
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer = ModelTrainer(config=model_trainer_config)
    model_trainer.train()
except Exception as e:
    from src.datascience import BidBotException
    import sys
    raise BidBotException(e, sys)

[ 2026-03-01 09:53:03,552 ] 30 datascience - INFO - yaml file: config\config.yaml loaded successfully
[ 2026-03-01 09:53:03,561 ] 30 datascience - INFO - yaml file: params.yaml loaded successfully
[ 2026-03-01 09:53:03,567 ] 30 datascience - INFO - yaml file: schema.yaml loaded successfully
[ 2026-03-01 09:53:03,570 ] 50 datascience - INFO - created directory at: artifacts
[ 2026-03-01 09:53:03,572 ] 50 datascience - INFO - created directory at: artifacts/model_trainer
[ 2026-03-01 09:53:03,590 ] 17 datascience - INFO - neg: 1528 | pos: 82 | scale_pos_weight: 18.63
[ 2026-03-01 09:53:03,595 ] 30 datascience - INFO - yaml file: params.yaml loaded successfully
[ 2026-03-01 09:53:03,599 ] 44 datascience - INFO - Starting GridSearchCV


Fitting 5 folds for each of 108 candidates, totalling 540 fits


[ 2026-03-01 09:53:42,702 ] 47 datascience - INFO - Best params:     {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.8}
[ 2026-03-01 09:53:42,704 ] 48 datascience - INFO - Best CV ROC AUC: 0.9038
[ 2026-03-01 09:53:42,712 ] 52 datascience - INFO - Model saved -> artifacts\model_trainer\model.joblib


In [30]:
# --- Evaluate + Log to MLflow ---
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation = ModelEvaluation(config=model_evaluation_config)
    metrics = model_evaluation.evaluate(thresholds=[ 0.4, 0.45, 0.5, 0.55, 0.6])
    print(metrics)
except Exception as e:
    from src.datascience import BidBotException
    import sys
    raise BidBotException(e, sys)

[ 2026-03-01 10:12:52,771 ] 30 datascience - INFO - yaml file: config\config.yaml loaded successfully
[ 2026-03-01 10:12:52,778 ] 30 datascience - INFO - yaml file: params.yaml loaded successfully
[ 2026-03-01 10:12:52,783 ] 30 datascience - INFO - yaml file: schema.yaml loaded successfully
[ 2026-03-01 10:12:52,786 ] 50 datascience - INFO - created directory at: artifacts
[ 2026-03-01 10:12:52,788 ] 50 datascience - INFO - created directory at: artifacts/model_evaluation
2026/03/01 10:13:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/01 10:13:31 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run languid-moose-690 at: https://dagshub.com/fakhrulfaiz/bid-bot-detection.mlflow/#/experiments/0/runs/7421e8bfdfd549fba9c7465892af46b8
🧪 View experiment at: https://dagshub.com/fakhrulfaiz/bid-bot-detection.mlflow/#/experiments/0
{'roc_auc': 0.9068, 'threshold_results': {'threshold_0.4': {'accuracy': 0.8859, 'precision': 0.2881, 'recall': 0.8095, 'f1': 0.425, 'confusion_matrix': [[340, 42], [4, 17]]}, 'threshold_0.45': {'accuracy': 0.8958, 'precision': 0.2941, 'recall': 0.7143, 'f1': 0.4167, 'confusion_matrix': [[346, 36], [6, 15]]}, 'threshold_0.5': {'accuracy': 0.9082, 'precision': 0.3095, 'recall': 0.619, 'f1': 0.4127, 'confusion_matrix': [[353, 29], [8, 13]]}, 'threshold_0.55': {'accuracy': 0.9107, 'precision': 0.3077, 'recall': 0.5714, 'f1': 0.4, 'confusion_matrix': [[355, 27], [9, 12]]}, 'threshold_0.6': {'accuracy': 0.9181, 'precision': 0.3235, 'recall': 0.5238, 'f1': 0.4, 'confusion_matrix': [[359, 23], [10, 11]]}}}
